# TinyRAG Colab CUDA Demo

This notebook clones the TinyRAG repository, installs the CUDA llama.cpp runtime with `uv`, downloads the configured GGUF model, rebuilds the local RAG index, and runs grounded generation with GPU offload.

Before running all cells, set **Runtime > Change runtime type > GPU** in Colab.

## Configuration

Change `MODEL_REPO`, `MODEL_FILE`, and the runtime settings here to try a different GGUF model. The configuration cell also keeps the previous Qwen2.5-1.5B baseline and Gemma 4 E2B candidate commented out for quick switching. Keep `N_CTX` and `N_GPU_LAYERS` conservative until VRAM usage is confirmed. Keep decoding settings consistent between ask and benchmark cells when comparing answer quality.

In [ ]:
REPO_URL = "https://github.com/cxyfer/tinyrag-laptop-assistant.git"
BRANCH = "main"
PROJECT_DIR = "/content/tinyrag-laptop-assistant"

MODEL_REPO = "bartowski/Qwen_Qwen3.5-2B-GGUF"
MODEL_FILE = "Qwen_Qwen3.5-2B-Q4_K_M.gguf"
# MODEL_REPO = "Qwen/Qwen2.5-1.5B-Instruct-GGUF"
# MODEL_FILE = "qwen2.5-1.5b-instruct-q4_k_m.gguf"
# MODEL_REPO = "unsloth/gemma-4-E2B-it-GGUF"
# MODEL_FILE = "gemma-4-E2B-it-Q4_K_M.gguf"
MODEL_PATH = f"models/{MODEL_FILE}"

QUESTION = "BXH 使用哪一張顯示卡？"
N_CTX = 2048
TEMPERATURE = 0.3
REPEAT_PENALTY = 1.15
FREQUENCY_PENALTY = 0.3
MAX_TOKENS = 256
BENCHMARK_MAX_TOKENS = 96
N_GPU_LAYERS = 35

## Check the GPU runtime

In [ ]:
!nvidia-smi

## Clone the repository

In [ ]:
!rm -rf {PROJECT_DIR}
!git clone --branch {BRANCH} {REPO_URL} {PROJECT_DIR}
%cd {PROJECT_DIR}

## Install the CUDA llama.cpp runtime

In [ ]:
!python -m pip install -q uv
!uv sync --frozen --extra llama-cu121

In [ ]:
GPU_CHECK = (
    "from llama_cpp import llama_cpp; "
    "print('gpu_offload_supported=', llama_cpp.llama_supports_gpu_offload())"
)
!uv run --frozen --extra llama-cu121 python -c "{GPU_CHECK}"

## Download the GGUF model

In [ ]:
!uvx --from huggingface-hub hf download {MODEL_REPO} {MODEL_FILE} --local-dir models

## Build the local RAG index

In [ ]:
!uv run --frozen --extra llama-cu121 tinyrag ingest --prefer-cache
!uv run --frozen --extra llama-cu121 tinyrag build-index

## Ask a grounded question

In [ ]:
ASK_CMD = (
    f'uv run --frozen --extra llama-cu121 tinyrag ask "{QUESTION}" '
    f"--model-path {MODEL_PATH} --n-ctx {N_CTX} "
    f"--temperature {TEMPERATURE} --max-tokens {MAX_TOKENS} "
    f"--repeat-penalty {REPEAT_PENALTY} --frequency-penalty {FREQUENCY_PENALTY} "
    f"--n-gpu-layers {N_GPU_LAYERS}"
)
!{ASK_CMD}

## Optional: run the real-model benchmark

Use the same decoding penalty settings as the ask cell so answer-quality comparisons are meaningful.

In [ ]:
BENCHMARK_CMD = (
    "uv run --frozen --extra llama-cu121 tinyrag benchmark --use-model "
    f"--model-path {MODEL_PATH} --n-ctx {N_CTX} "
    f"--temperature {TEMPERATURE} --max-tokens {BENCHMARK_MAX_TOKENS} "
    f"--repeat-penalty {REPEAT_PENALTY} --frequency-penalty {FREQUENCY_PENALTY} "
    f"--n-gpu-layers {N_GPU_LAYERS}"
)
!{BENCHMARK_CMD}